Export libraries

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler


Export data

In [2]:
train_data = pd.read_csv("data/bank_data_train.csv")
test_data = pd.read_csv("data/bank_data_test.csv")

In [3]:
print(f'Train_shape: {train_data.shape}')
print(f'Test_shape: {test_data.shape}')

Train_shape: (355190, 116)
Test_shape: (88798, 116)


In [4]:
train_data.sample(5)

,ID,CR_PROD_CNT_IL,AMOUNT_RUB_CLO_PRC,PRC_ACCEPTS_A_EMAIL_LINK,APP_REGISTR_RGN_CODE,PRC_ACCEPTS_A_POS,PRC_ACCEPTS_A_TK,TURNOVER_DYNAMIC_IL_1M,CNT_TRAN_AUT_TENDENCY1M,SUM_TRAN_AUT_TENDENCY1M,...,REST_DYNAMIC_CC_3M,MED_DEBT_PRC_YWZ,LDEAL_ACT_DAYS_PCT_TR3,LDEAL_ACT_DAYS_PCT_AAVG,LDEAL_DELINQ_PER_MAXYWZ,TURNOVER_DYNAMIC_CC_3M,LDEAL_ACT_DAYS_PCT_TR,LDEAL_ACT_DAYS_PCT_TR4,LDEAL_ACT_DAYS_PCT_CURR,TARGET
341242,573412,0,0.022929,NaN,NaN,NaN,NaN,0.0,0.250000,0.220557,...,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0
209841,408885,0,1.000000,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0
323922,551776,0,0.024723,0.0,NaN,0.0,0.0,0.0,0.304348,0.168558,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
307980,531811,0,0.000000,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0
64649,227675,0,0.032744,0.0,NaN,0.0,0.0,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [5]:
test_data.sample(5)

,ID,CR_PROD_CNT_IL,AMOUNT_RUB_CLO_PRC,PRC_ACCEPTS_A_EMAIL_LINK,APP_REGISTR_RGN_CODE,PRC_ACCEPTS_A_POS,PRC_ACCEPTS_A_TK,TURNOVER_DYNAMIC_IL_1M,CNT_TRAN_AUT_TENDENCY1M,SUM_TRAN_AUT_TENDENCY1M,...,REST_DYNAMIC_CC_3M,MED_DEBT_PRC_YWZ,LDEAL_ACT_DAYS_PCT_TR3,LDEAL_ACT_DAYS_PCT_AAVG,LDEAL_DELINQ_PER_MAXYWZ,TURNOVER_DYNAMIC_CC_3M,LDEAL_ACT_DAYS_PCT_TR,LDEAL_ACT_DAYS_PCT_TR4,LDEAL_ACT_DAYS_PCT_CURR,TARGET
39,369635,0,0.174559,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
48662,231052,0,0.069734,NaN,NaN,NaN,NaN,0.0,0.333333,0.273619,...,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
36624,563782,0,0.000000,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
54978,242319,0,0.250150,0.0,NaN,0.0,0.0,0.0,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN
67901,477102,0,0.000000,NaN,NaN,NaN,NaN,0.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN


In [6]:
print(f"TARGET is 0: {train_data[train_data['TARGET']==0]['TARGET'].count()}")
print(f"TARGET is 1: {train_data[train_data['TARGET']==1]['TARGET'].count()}")

TARGET is 0: 326265
TARGET is 1: 28925


In [7]:
test_data.drop(columns={'TARGET'}, inplace=True)

Preprocessing

In [8]:
with pd.option_context('display.max_rows', None):
    print(train_data.isna().sum().sort_values(ascending=False))

CLNT_SALARY_VALUE              354478
LDEAL_YQZ_COM                  353950
LDEAL_YQZ_CHRG                 353949
AVG_PCT_MONTH_TO_PCLOSE        353562
MAX_PCLOSE_DATE                353309
LDEAL_AMT_MONTH                353302
AVG_PCT_DEBT_TO_DEAL_AMT       353302
LDEAL_YQZ_PC                   352382
LDEAL_DELINQ_PER_MAXYQZ        347189
LDEAL_TENOR_MIN                347189
MED_DEBT_PRC_YQZ               347189
DEAL_YQZ_IR_MIN                347189
LDEAL_USED_AMT_AVG_YQZ         347189
LDEAL_TENOR_MAX                347189
DEAL_YQZ_IR_MAX                347189
CLNT_JOB_POSITION_TYPE         310409
APP_CAR                        297934
APP_TRAVEL_PASS                297933
APP_DRIVING_LICENSE            297933
APP_KIND_OF_PROP_HABITATION    295829
APP_POSITION_TYPE              294645
APP_REGISTR_RGN_CODE           294640
SUM_TRAN_CLO_TENDENCY1M        288894
CNT_TRAN_CLO_TENDENCY1M        288894
APP_COMP_TYPE                  287828
APP_EMP_TYPE                   287828
APP_EDUCATIO

We have 4 groups of features with NaN:
1) Transaction activity (TRAN, TENDENCY, ACCEPTS, AMOUNT_RUB, TRANS_COUNT) → fillna(0)
2) Credit history / deals (LDEAL_*, DEAL_*, MED_DEBT_PRC_Y*) → fillna(0) + availability flag
3) Personal data (APP_*) → fillna('-1')
4) Сustomer demographic data → median/avg + flag is_missing

In [9]:
tran_cols = ['CNT_TRAN_MED_TENDENCY1M', 'SUM_TRAN_MED_TENDENCY1M',
'CNT_TRAN_AUT_TENDENCY1M', 'SUM_TRAN_AUT_TENDENCY1M',
'CNT_TRAN_AUT_TENDENCY3M', 'SUM_TRAN_AUT_TENDENCY3M',
'CNT_TRAN_CLO_TENDENCY3M', 'SUM_TRAN_CLO_TENDENCY3M',
'CNT_TRAN_CLO_TENDENCY1M', 'SUM_TRAN_CLO_TENDENCY1M',
'CNT_TRAN_MED_TENDENCY3M', 'SUM_TRAN_MED_TENDENCY3M',
'CNT_TRAN_SUP_TENDENCY1M', 'SUM_TRAN_SUP_TENDENCY1M',
'CNT_TRAN_SUP_TENDENCY3M', 'SUM_TRAN_SUP_TENDENCY3M',
'CNT_TRAN_ATM_TENDENCY1M', 'SUM_TRAN_ATM_TENDENCY1M',
'CNT_TRAN_ATM_TENDENCY3M', 'SUM_TRAN_ATM_TENDENCY3M',
'TRANS_CNT_TENDENCY3M', 'TRANS_AMOUNT_TENDENCY3M',
'AMOUNT_RUB_CLO_PRC', 'AMOUNT_RUB_SUP_PRC', 'AMOUNT_RUB_ATM_PRC', 'AMOUNT_RUB_NAS_PRC',
'TRANS_COUNT_ATM_PRC', 'TRANS_COUNT_NAS_PRC', 'TRANS_COUNT_SUP_PRC',
'PRC_ACCEPTS_A_AMOBILE', 'PRC_ACCEPTS_A_POS', 'PRC_ACCEPTS_A_MTP', 'PRC_ACCEPTS_TK',
'CNT_ACCEPTS_MTP', 'PRC_ACCEPTS_MTP', 'PRC_ACCEPTS_A_ATM', 'PRC_ACCEPTS_A_EMAIL_LINK',
'CNT_ACCEPTS_TK', 'PRC_ACCEPTS_A_TK']

loan_cols = ['LDEAL_YQZ_COM', 'LDEAL_YQZ_CHRG', 'LDEAL_YQZ_PC',
'AVG_PCT_MONTH_TO_PCLOSE', 'MAX_PCLOSE_DATE', 'LDEAL_AMT_MONTH', 'AVG_PCT_DEBT_TO_DEAL_AMT',
'LDEAL_DELINQ_PER_MAXYQZ', 'LDEAL_TENOR_MIN', 'LDEAL_TENOR_MAX',
'MED_DEBT_PRC_YQZ', 'DEAL_YQZ_IR_MIN', 'DEAL_YQZ_IR_MAX', 'LDEAL_USED_AMT_AVG_YQZ',
'DEAL_GRACE_DAYS_ACC_MAX', 'DEAL_GRACE_DAYS_ACC_AVG', 'DEAL_GRACE_DAYS_ACC_S1X1',
'LDEAL_ACT_DAYS_PCT_TR4', 'LDEAL_ACT_DAYS_PCT_TR', 'LDEAL_ACT_DAYS_ACC_PCT_AVG',
'LDEAL_ACT_DAYS_PCT_CURR', 'LDEAL_ACT_DAYS_PCT_TR3', 'LDEAL_ACT_DAYS_PCT_AAVG',
'LDEAL_USED_AMT_AVG_YWZ', 'DEAL_YWZ_IR_MIN', 'DEAL_YWZ_IR_MAX',
'MED_DEBT_PRC_YWZ', 'LDEAL_DELINQ_PER_MAXYWZ']

train_data['has_loan_yqz'] = train_data['LDEAL_YQZ_COM'].notna().astype(int)
train_data['has_loan_ywz'] = train_data['LDEAL_USED_AMT_AVG_YWZ'].notna().astype(int)

app_cols = ['APP_CAR', 'APP_TRAVEL_PASS', 'APP_DRIVING_LICENSE',
'APP_KIND_OF_PROP_HABITATION', 'APP_POSITION_TYPE', 'APP_REGISTR_RGN_CODE',
'APP_COMP_TYPE', 'APP_EMP_TYPE', 'APP_EDUCATION', 'APP_MARITAL_STATUS']

clnt_cat_cols = ['CLNT_JOB_POSITION_TYPE', 'CLNT_JOB_POSITION', 'CLNT_TRUST_RELATION']

/var/folders/hp/wkxw2f2d3n54s_jrt26bd91r0000gn/T/ipykernel_90081/3797702133.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_data['has_loan_yqz'] = train_data['LDEAL_YQZ_COM'].notna().astype(int)
/var/folders/hp/wkxw2f2d3n54s_jrt26bd91r0000gn/T/ipykernel_90081/3797702133.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_data['has_loan_ywz'] = train_data['LDEAL_USED_AMT_AVG_YWZ'].notna().astype(int)


In [10]:
train_data[tran_cols] = train_data[tran_cols].fillna(0)

yqz_cols = [c for c in loan_cols if 'YQZ' in c]
ywz_cols = [c for c in loan_cols if 'YWZ' in c]
other_loan_cols = [c for c in loan_cols if c not in yqz_cols + ywz_cols]

train_data['has_loan_yqz'] = train_data[yqz_cols].notna().any(axis=1).astype(int)
train_data['has_loan_ywz'] = train_data[ywz_cols].notna().any(axis=1).astype(int)

train_data[loan_cols] = train_data[loan_cols].fillna(0)
train_data[app_cols] = train_data[app_cols].fillna(-1)
train_data[clnt_cat_cols] = train_data[clnt_cat_cols].fillna(-1)

train_data['salary_is_missing'] = train_data['CLNT_SALARY_VALUE'].isna().astype(int)
group_medians = train_data.groupby('CLNT_JOB_POSITION_TYPE')['CLNT_SALARY_VALUE'].transform('median')
global_median = train_data['CLNT_SALARY_VALUE'].median()
train_data['CLNT_SALARY_VALUE'] = (
    train_data['CLNT_SALARY_VALUE']
    .fillna(group_medians)
    .fillna(global_median)
)

/var/folders/hp/wkxw2f2d3n54s_jrt26bd91r0000gn/T/ipykernel_90081/667628208.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_data['salary_is_missing'] = train_data['CLNT_SALARY_VALUE'].isna().astype(int)


In [107]:
test_data[tran_cols] = test_data[tran_cols].fillna(0)

test_data['has_loan_yqz'] = test_data[yqz_cols].notna().any(axis=1).astype(int)
test_data['has_loan_ywz'] = test_data[ywz_cols].notna().any(axis=1).astype(int)
test_data[loan_cols] = test_data[loan_cols].fillna(0)

test_data[app_cols] = test_data[app_cols].fillna(-1)

test_data[clnt_cat_cols] = test_data[clnt_cat_cols].fillna(-1)

test_data['salary_is_missing'] = test_data['CLNT_SALARY_VALUE'].isna().astype(int)

test_group_medians = test_data['CLNT_JOB_POSITION_TYPE'].map(
    train_data.groupby('CLNT_JOB_POSITION_TYPE')['CLNT_SALARY_VALUE'].median()
)
test_data['CLNT_SALARY_VALUE'] = (
    test_data['CLNT_SALARY_VALUE']
    .fillna(test_group_medians)
    .fillna(global_median)
)

/var/folders/hp/wkxw2f2d3n54s_jrt26bd91r0000gn/T/ipykernel_39085/1308087204.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_data['has_loan_yqz'] = test_data[yqz_cols].notna().any(axis=1).astype(int)
/var/folders/hp/wkxw2f2d3n54s_jrt26bd91r0000gn/T/ipykernel_39085/1308087204.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_data['has_loan_ywz'] = test_data[ywz_cols].notna().any(axis=1).astype(int)
/var/folders/hp/wkxw2f2d3n54s_jrt26bd91r0000gn/T/ipykernel_39085/1308087204.py:11: PerformanceWarning: DataFrame is hi

In [11]:
X = train_data.drop(columns=['TARGET'])
y = train_data['TARGET']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_val.shape)
print(y_train.value_counts(normalize=True))
print(y_val.value_counts(normalize=True))

(284152, 118) (71038, 118)
TARGET
0    0.918565
1    0.081435
Name: proportion, dtype: float64
TARGET
0    0.918565
1    0.081435
Name: proportion, dtype: float64


In [13]:
non_numeric = X_train.select_dtypes(include=['object', 'string']).columns.tolist()
print(non_numeric)

['CLNT_TRUST_RELATION', 'APP_MARITAL_STATUS', 'APP_KIND_OF_PROP_HABITATION', 'CLNT_JOB_POSITION_TYPE', 'CLNT_JOB_POSITION', 'APP_DRIVING_LICENSE', 'APP_EDUCATION', 'APP_TRAVEL_PASS', 'APP_CAR', 'APP_POSITION_TYPE', 'APP_EMP_TYPE', 'APP_COMP_TYPE', 'PACK']


In [14]:
cat_cols_to_encode = ['CLNT_TRUST_RELATION', 'APP_MARITAL_STATUS', 'APP_KIND_OF_PROP_HABITATION',
                       'CLNT_JOB_POSITION_TYPE', 'CLNT_JOB_POSITION', 'APP_DRIVING_LICENSE',
                       'APP_EDUCATION', 'APP_TRAVEL_PASS', 'APP_CAR', 'APP_POSITION_TYPE',
                       'APP_EMP_TYPE', 'APP_COMP_TYPE', 'PACK']

for col in cat_cols_to_encode:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)

label_encoders = {}
for col in cat_cols_to_encode:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    known_classes = set(le.classes_)
    X_val[col] = X_val[col].apply(lambda x: x if x in known_classes else 'unseen')
    if 'unseen' not in le.classes_:
        le.classes_ = np.append(le.classes_, 'unseen')
    X_val[col] = le.transform(X_val[col])
    label_encoders[col] = le

print(X_train.dtypes.value_counts())
print(X_val.dtypes.value_counts())

float64    94
int64      24
Name: count, dtype: int64
float64    94
int64      24
Name: count, dtype: int64


In [15]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

Models plan:  
1) Baseline - naive classifier  
2) Random forest  
3) Sklearn MLPClassifier  
4) Keras (expect same result like sklearn library)  
5) Tensorflow (expect same result like sklearn library)  
6) Numpy (expect same result like sklearn library)

1. Baseline - naive classifier

In [16]:
baseline = DummyClassifier(strategy='stratified', random_state=42)
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)
baseline_proba = baseline.predict_proba(X_val)[:, 1]

baseline_acc = accuracy_score(y_val, baseline_pred)
baseline_auc = roc_auc_score(y_val, baseline_proba)

print(f"Baseline Accuracy: {baseline_acc:.4f}")
print(f"Baseline AUC: {baseline_auc:.4f}")

Baseline Accuracy: 0.8514
Baseline AUC: 0.5002


2. Random forest

In [17]:
rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 10],
    'class_weight': ['balanced']
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

rf_grid = GridSearchCV(
    rf, rf_param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2
)

rf_grid.fit(X_train, y_train)

print("Best params:", rf_grid.best_params_)
print("Best CV AUC:", rf_grid.best_score_)

Fitting 3 folds for each of 12 candidates, totalling 36 fits
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=100; total time=  30.6s
[CV] END class_weight=balanced, max_depth=10, min_samples_split=10, n_estimators=100; total time=  31.0s
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=100; total time=  31.1s
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=100; total time=  31.3s
[CV] END class_weight=balanced, max_depth=10, min_samples_split=10, n_estimators=100; total time=  31.4s
[CV] END class_weight=balanced, max_depth=10, min_samples_split=10, n_estimators=100; total time=  31.7s
[CV] END class_weight=balanced, max_depth=10, min_samples_split=10, n_estimators=200; total time=  59.9s
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=200; total time= 1.0min
[CV] END class_weight=balanced, max_depth=10, min_samples_split=2, n_estimators=200; total time= 1.0min

In [18]:
best_rf = rf_grid.best_estimator_
rf_pred = best_rf.predict(X_val)
rf_proba = best_rf.predict_proba(X_val)[:, 1]

rf_acc = accuracy_score(y_val, rf_pred)
rf_auc = roc_auc_score(y_val, rf_proba)

print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"Random Forest AUC: {rf_auc:.4f}")

Random Forest Accuracy: 0.8974
Random Forest AUC: 0.8356


3. Sklearn MLPClassifier

In [19]:
mlp_param_grid = {
    'hidden_layer_sizes': [(64,), (128, 64), (128, 64, 32)],
    'alpha': [0.0001, 0.001, 0.01],
    'learning_rate_init': [0.001, 0.01],
}

mlp = MLPClassifier(
    max_iter=200,
    early_stopping=True,
    random_state=42
)

mlp_grid = GridSearchCV(
    mlp, mlp_param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=2
)

mlp_grid.fit(X_train_scaled, y_train)

print("Best params:", mlp_grid.best_params_)
print("Best CV AUC:", mlp_grid.best_score_)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=   6.8s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=   8.4s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.01; total time=   8.6s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.001; total time=   9.0s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.01; total time=   9.1s
[CV] END alpha=0.0001, hidden_layer_sizes=(128, 64), learning_rate_init=0.01; total time=  10.9s
[CV] END alpha=0.0001, hidden_layer_sizes=(64,), learning_rate_init=0.01; total time=  12.4s
[CV] END alpha=0.0001, hidden_layer_sizes=(128, 64), learning_rate_init=0.001; total time=  14.1s
[CV] END alpha=0.0001, hidden_layer_sizes=(128, 64), learning_rate_init=0.001; total time=  15.4s
[CV] END alpha=0.0001, hidden_layer_sizes=(128, 64), learning_rate_init=0.001; total 

In [20]:
best_mlp = mlp_grid.best_estimator_
mlp_pred = best_mlp.predict(X_val_scaled)
mlp_proba = best_mlp.predict_proba(X_val_scaled)[:, 1]

mlp_acc = accuracy_score(y_val, mlp_pred)
mlp_auc = roc_auc_score(y_val, mlp_proba)

print(f"MLPClassifier Accuracy: {mlp_acc:.4f}")
print(f"MLPClassifier AUC: {mlp_auc:.4f}")

MLPClassifier Accuracy: 0.9186
MLPClassifier AUC: 0.8004


4. Keras

In [21]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
print(class_weight_dict)

{0: np.float64(0.5443274638713929), 1: np.float64(6.139844425237683)}


In [22]:
class ChurnNeuralNetwork:
    def __init__(self, input_dim, hidden_layers=(128, 64), dropout_rate=0.3, learning_rate=0.001):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.model = self._build_model()

    def _build_model(self):
        model = keras.Sequential()
        model.add(keras.layers.Input(shape=(self.input_dim,)))
        for units in self.hidden_layers:
            model.add(keras.layers.Dense(units, activation='relu'))
            model.add(keras.layers.Dropout(self.dropout_rate))
        model.add(keras.layers.Dense(1, activation='sigmoid'))

        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=self.learning_rate),
            loss='binary_crossentropy',
            metrics=['accuracy', keras.metrics.AUC(name='auc')]
        )
        return model

    def fit(self, X_train, y_train, X_val, y_val, class_weight=None, epochs=50, batch_size=256):
        early_stop = keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=5, restore_best_weights=True
        )
        history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            class_weight=class_weight,
            callbacks=[early_stop],
            verbose=1
        )
        return history

    def predict_proba(self, X):
        return self.model.predict(X).flatten()

    def evaluate(self, X, y):
        proba = self.predict_proba(X)
        pred = (proba >= 0.5).astype(int)
        acc = accuracy_score(y, pred)
        auc = roc_auc_score(y, proba)
        return acc, auc

In [23]:
nn = ChurnNeuralNetwork(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=(128, 64),
    dropout_rate=0.3,
    learning_rate=0.001
)

history = nn.fit(
    X_train_scaled, y_train.values,
    X_val_scaled, y_val.values,
    class_weight=class_weight_dict,
    epochs=50,
    batch_size=256
)

nn_acc, nn_auc = nn.evaluate(X_val_scaled, y_val.values)
print(f"Keras NN Accuracy: {nn_acc:.4f}")
print(f"Keras NN AUC: {nn_auc:.4f}")

Epoch 1/50
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 1s 850us/step - accuracy: 0.6197 - auc: 0.7070 - loss: 0.6270 - val_accuracy: 0.6365 - val_auc: 0.7560 - val_loss: 0.5898
Epoch 2/50
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 1s 777us/step - accuracy: 0.6399 - auc: 0.7526 - loss: 0.5878 - val_accuracy: 0.6578 - val_auc: 0.7708 - val_loss: 0.5585
Epoch 3/50
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 1s 773us/step - accuracy: 0.6455 - auc: 0.7674 - loss: 0.5718 - val_accuracy: 0.6364 - val_auc: 0.7831 - val_loss: 0.5751
Epoch 4/50
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 1s 764us/step - accuracy: 0.6463 - auc: 0.7766 - loss: 0.5606 - val_accuracy: 0.6519 - val_auc: 0.7891 - val_loss: 0.5523
Epoch 5/50
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 1s 758us/step - accuracy: 0.6496 - auc: 0.7836 - loss: 0.5534 - val_accuracy: 0.6510 - val_auc: 0.7914 - val_loss: 0.5582
Epoch 6/50
1110/1110 ━━━━━━━━━━━━━━━━━━━━ 1s 777us/step - accuracy: 0.6541 - auc: 0.7890 - loss: 0.5466 - val_accuracy: 0.6492 - val_auc: 0.7986 - val_loss: 0.5553
Epoch 7/50
1110/

5. Tensorflow

In [24]:
class ChurnTensorFlowNN:
    def __init__(self, input_dim, hidden_layers=(128, 64), dropout_rate=0.3, learning_rate=0.001):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.dropout_rate = dropout_rate
        self.learning_rate = learning_rate
        self.weights = []
        self.biases = []
        self._init_params()
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)

    def _init_params(self):
        dims = [self.input_dim] + list(self.hidden_layers) + [1]
        for i in range(len(dims) - 1):
            w = tf.Variable(
                tf.random.normal([dims[i], dims[i+1]], stddev=tf.sqrt(2.0 / dims[i])),
                trainable=True, name=f'W{i}'
            )
            b = tf.Variable(tf.zeros([dims[i+1]]), trainable=True, name=f'b{i}')
            self.weights.append(w)
            self.biases.append(b)

    def _forward(self, X, training=True):
        a = X
        n_layers = len(self.weights)
        for i in range(n_layers - 1):
            z = tf.matmul(a, self.weights[i]) + self.biases[i]
            a = tf.nn.relu(z)
            if training:
                a = tf.nn.dropout(a, rate=self.dropout_rate)
        z_out = tf.matmul(a, self.weights[-1]) + self.biases[-1]
        return tf.nn.sigmoid(z_out)

    def _loss(self, y_true, y_pred, sample_weight=None):
        y_true = tf.reshape(tf.cast(y_true, tf.float32), (-1, 1))
        bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
        if sample_weight is not None:
            bce = bce * sample_weight
        return tf.reduce_mean(bce)

    def _train_step(self, X_batch, y_batch, sample_weight_batch):
        with tf.GradientTape() as tape:
            y_pred = self._forward(X_batch, training=True)
            loss = self._loss(y_batch, y_pred, sample_weight_batch)
        variables = self.weights + self.biases
        grads = tape.gradient(loss, variables)
        self.optimizer.apply_gradients(zip(grads, variables))
        return loss

    def fit(self, X_train, y_train, X_val, y_val, class_weight=None,
            epochs=50, batch_size=256, patience=5, verbose=True):
        X_train = tf.constant(X_train, dtype=tf.float32)
        y_train = tf.constant(y_train, dtype=tf.float32)

        if class_weight is not None:
            sample_weights = np.where(y_train.numpy() == 1, class_weight[1], class_weight[0])
            sample_weights = tf.constant(sample_weights, dtype=tf.float32)
        else:
            sample_weights = tf.ones_like(y_train)

        n = X_train.shape[0]
        best_val_auc = -np.inf
        best_weights = None
        wait = 0

        for epoch in range(epochs):
            idx = tf.random.shuffle(tf.range(n))
            X_shuf = tf.gather(X_train, idx)
            y_shuf = tf.gather(y_train, idx)
            w_shuf = tf.gather(sample_weights, idx)

            epoch_loss = 0.0
            n_batches = 0
            for start in range(0, n, batch_size):
                end = start + batch_size
                loss = self._train_step(X_shuf[start:end], y_shuf[start:end], w_shuf[start:end])
                epoch_loss += loss.numpy()
                n_batches += 1

            val_proba = self.predict_proba(X_val)
            val_auc = roc_auc_score(y_val, val_proba)

            if verbose:
                print(f"Epoch {epoch+1}/{epochs} - loss: {epoch_loss/n_batches:.4f} - val_auc: {val_auc:.4f}")

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_weights = [w.numpy().copy() for w in self.weights]
                best_biases = [b.numpy().copy() for b in self.biases]
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(f"Early stopping at epoch {epoch+1}, best val_auc: {best_val_auc:.4f}")
                    break

        for i in range(len(self.weights)):
            self.weights[i].assign(best_weights[i])
            self.biases[i].assign(best_biases[i])

    def predict_proba(self, X):
        X = tf.constant(X, dtype=tf.float32)
        proba = self._forward(X, training=False)
        return proba.numpy().flatten()

    def evaluate(self, X, y):
        proba = self.predict_proba(X)
        pred = (proba >= 0.5).astype(int)
        acc = accuracy_score(y, pred)
        auc = roc_auc_score(y, proba)
        return acc, auc

In [25]:
tf_nn = ChurnTensorFlowNN(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=(128, 64),
    dropout_rate=0.3,
    learning_rate=0.001
)

tf_nn.fit(
    X_train_scaled, y_train.values,
    X_val_scaled, y_val.values,
    class_weight=class_weight_dict,
    epochs=50,
    batch_size=256,
    patience=5
)

tf_acc, tf_auc = tf_nn.evaluate(X_val_scaled, y_val.values)
print(f"TensorFlow NN Accuracy: {tf_acc:.4f}")
print(f"TensorFlow NN AUC: {tf_auc:.4f}")

Epoch 1/50 - loss: 0.6371 - val_auc: 0.7532
Epoch 2/50 - loss: 0.5911 - val_auc: 0.7685
Epoch 3/50 - loss: 0.5746 - val_auc: 0.7757
Epoch 4/50 - loss: 0.5649 - val_auc: 0.7840
Epoch 5/50 - loss: 0.5562 - val_auc: 0.7908
Epoch 6/50 - loss: 0.5492 - val_auc: 0.7967
Epoch 7/50 - loss: 0.5446 - val_auc: 0.8022
Epoch 8/50 - loss: 0.5371 - val_auc: 0.8042
Epoch 9/50 - loss: 0.5351 - val_auc: 0.8054
Epoch 10/50 - loss: 0.5319 - val_auc: 0.8093
Epoch 11/50 - loss: 0.5275 - val_auc: 0.8110
Epoch 12/50 - loss: 0.5245 - val_auc: 0.8129
Epoch 13/50 - loss: 0.5220 - val_auc: 0.8101
Epoch 14/50 - loss: 0.5207 - val_auc: 0.8152
Epoch 15/50 - loss: 0.5174 - val_auc: 0.8161
Epoch 16/50 - loss: 0.5145 - val_auc: 0.8175
Epoch 17/50 - loss: 0.5129 - val_auc: 0.8193
Epoch 18/50 - loss: 0.5102 - val_auc: 0.8201
Epoch 19/50 - loss: 0.5100 - val_auc: 0.8201
Epoch 20/50 - loss: 0.5092 - val_auc: 0.8204
Epoch 21/50 - loss: 0.5071 - val_auc: 0.8213
Epoch 22/50 - loss: 0.5062 - val_auc: 0.8202
Epoch 23/50 - loss:

6. Numpy

In [26]:
class ChurnNumpyNN:
    def __init__(self, input_dim, hidden_layers=(128, 64), dropout_rate=0.3, learning_rate=0.001, seed=42):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.dropout_rate = dropout_rate
        self.lr = learning_rate
        rng = np.random.default_rng(seed)

        dims = [input_dim] + list(hidden_layers) + [1]
        self.W = []
        self.b = []
        for i in range(len(dims) - 1):
            w = rng.normal(0, np.sqrt(2.0 / dims[i]), size=(dims[i], dims[i+1]))
            b = np.zeros((1, dims[i+1]))
            self.W.append(w)
            self.b.append(b)

        self.mW = [np.zeros_like(w) for w in self.W]
        self.vW = [np.zeros_like(w) for w in self.W]
        self.mb = [np.zeros_like(b) for b in self.b]
        self.vb = [np.zeros_like(b) for b in self.b]
        self.t = 0
        self.beta1, self.beta2, self.eps = 0.9, 0.999, 1e-8

    @staticmethod
    def _relu(z):
        return np.maximum(0, z)

    @staticmethod
    def _relu_deriv(z):
        return (z > 0).astype(float)

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def _forward(self, X, training=True, rng=None):
        cache = {'A0': X}
        A = X
        n_layers = len(self.W)
        for i in range(n_layers - 1):
            Z = A @ self.W[i] + self.b[i]
            A = self._relu(Z)
            if training:
                mask = (rng.random(A.shape) > self.dropout_rate).astype(float)
                A = A * mask / (1 - self.dropout_rate)
                cache[f'mask{i}'] = mask
            cache[f'Z{i+1}'] = Z
            cache[f'A{i+1}'] = A
        Z_out = A @ self.W[-1] + self.b[-1]
        A_out = self._sigmoid(Z_out)
        cache[f'Z{n_layers}'] = Z_out
        cache[f'A{n_layers}'] = A_out
        return A_out, cache

    def _backward(self, y, cache, sample_weight):
        n_layers = len(self.W)
        m = y.shape[0]
        y = y.reshape(-1, 1)
        sample_weight = sample_weight.reshape(-1, 1)

        grads_W = [None] * n_layers
        grads_b = [None] * n_layers

        A_out = cache[f'A{n_layers}']
        dZ = (A_out - y) * sample_weight / m

        for i in reversed(range(n_layers)):
            A_prev = cache[f'A{i}'] if i > 0 else cache['A0']
            grads_W[i] = A_prev.T @ dZ
            grads_b[i] = np.sum(dZ, axis=0, keepdims=True)

            if i > 0:
                dA_prev = dZ @ self.W[i].T
                if f'mask{i-1}' in cache:
                    dA_prev = dA_prev * cache[f'mask{i-1}'] / (1 - self.dropout_rate)
                dZ = dA_prev * self._relu_deriv(cache[f'Z{i}'])

        return grads_W, grads_b

    def _adam_step(self, grads_W, grads_b):
        self.t += 1
        for i in range(len(self.W)):
            self.mW[i] = self.beta1 * self.mW[i] + (1 - self.beta1) * grads_W[i]
            self.vW[i] = self.beta2 * self.vW[i] + (1 - self.beta2) * (grads_W[i] ** 2)
            mW_hat = self.mW[i] / (1 - self.beta1 ** self.t)
            vW_hat = self.vW[i] / (1 - self.beta2 ** self.t)
            self.W[i] -= self.lr * mW_hat / (np.sqrt(vW_hat) + self.eps)

            self.mb[i] = self.beta1 * self.mb[i] + (1 - self.beta1) * grads_b[i]
            self.vb[i] = self.beta2 * self.vb[i] + (1 - self.beta2) * (grads_b[i] ** 2)
            mb_hat = self.mb[i] / (1 - self.beta1 ** self.t)
            vb_hat = self.vb[i] / (1 - self.beta2 ** self.t)
            self.b[i] -= self.lr * mb_hat / (np.sqrt(vb_hat) + self.eps)

    def fit(self, X_train, y_train, X_val, y_val, class_weight=None,
            epochs=50, batch_size=256, patience=5, seed=42, verbose=True):
        rng = np.random.default_rng(seed)
        n = X_train.shape[0]

        if class_weight is not None:
            sw = np.where(y_train == 1, class_weight[1], class_weight[0])
        else:
            sw = np.ones(n)

        best_val_auc = -np.inf
        best_state = None
        wait = 0

        for epoch in range(epochs):
            perm = rng.permutation(n)
            X_shuf, y_shuf, sw_shuf = X_train[perm], y_train[perm], sw[perm]

            epoch_loss = 0.0
            n_batches = 0
            for start in range(0, n, batch_size):
                end = start + batch_size
                Xb, yb, swb = X_shuf[start:end], y_shuf[start:end], sw_shuf[start:end]

                A_out, cache = self._forward(Xb, training=True, rng=rng)
                eps = 1e-7
                bce = -(yb.reshape(-1,1) * np.log(A_out + eps) + (1 - yb.reshape(-1,1)) * np.log(1 - A_out + eps))
                loss = np.mean(bce.flatten() * swb)
                epoch_loss += loss
                n_batches += 1

                grads_W, grads_b = self._backward(yb, cache, swb)
                self._adam_step(grads_W, grads_b)

            val_proba = self.predict_proba(X_val)
            val_auc = roc_auc_score(y_val, val_proba)

            if verbose:
                print(f"Epoch {epoch+1}/{epochs} - loss: {epoch_loss/n_batches:.4f} - val_auc: {val_auc:.4f}")

            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_state = ([w.copy() for w in self.W], [b.copy() for b in self.b])
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(f"Early stopping at epoch {epoch+1}, best val_auc: {best_val_auc:.4f}")
                    break

        self.W, self.b = best_state

    def predict_proba(self, X):
        proba, _ = self._forward(X, training=False, rng=None)
        return proba.flatten()

    def evaluate(self, X, y):
        proba = self.predict_proba(X)
        pred = (proba >= 0.5).astype(int)
        acc = accuracy_score(y, pred)
        auc = roc_auc_score(y, proba)
        return acc, auc

In [27]:
np_nn = ChurnNumpyNN(
    input_dim=X_train_scaled.shape[1],
    hidden_layers=(128, 64),
    dropout_rate=0.3,
    learning_rate=0.001
)

np_nn.fit(
    X_train_scaled, y_train.values,
    X_val_scaled, y_val.values,
    class_weight=class_weight_dict,
    epochs=50,
    batch_size=256,
    patience=5
)

np_acc, np_auc = np_nn.evaluate(X_val_scaled, y_val.values)
print(f"NumPy NN Accuracy: {np_acc:.4f}")
print(f"NumPy NN AUC: {np_auc:.4f}")

Epoch 1/50 - loss: 0.6487 - val_auc: 0.7423
Epoch 2/50 - loss: 0.5969 - val_auc: 0.7661
Epoch 3/50 - loss: 0.5801 - val_auc: 0.7775
Epoch 4/50 - loss: 0.5686 - val_auc: 0.7841
Epoch 5/50 - loss: 0.5594 - val_auc: 0.7894
Epoch 6/50 - loss: 0.5515 - val_auc: 0.7950
Epoch 7/50 - loss: 0.5439 - val_auc: 0.7989
Epoch 8/50 - loss: 0.5403 - val_auc: 0.8034
Epoch 9/50 - loss: 0.5370 - val_auc: 0.8069
Epoch 10/50 - loss: 0.5312 - val_auc: 0.8103
Epoch 11/50 - loss: 0.5279 - val_auc: 0.8134
Epoch 12/50 - loss: 0.5245 - val_auc: 0.8131
Epoch 13/50 - loss: 0.5218 - val_auc: 0.8146
Epoch 14/50 - loss: 0.5195 - val_auc: 0.8159
Epoch 15/50 - loss: 0.5178 - val_auc: 0.8164
Epoch 16/50 - loss: 0.5148 - val_auc: 0.8173
Epoch 17/50 - loss: 0.5129 - val_auc: 0.8180
Epoch 18/50 - loss: 0.5123 - val_auc: 0.8183
Epoch 19/50 - loss: 0.5101 - val_auc: 0.8206
Epoch 20/50 - loss: 0.5080 - val_auc: 0.8178
Epoch 21/50 - loss: 0.5076 - val_auc: 0.8209
Epoch 22/50 - loss: 0.5066 - val_auc: 0.8225
Epoch 23/50 - loss:

In [28]:
results = pd.DataFrame([
    {
        'Library': 'sklearn',
        'Algorithm': 'DummyClassifier (baseline)',
        'Hyperparameters': "strategy='stratified'",
        'Accuracy': baseline_acc,
        'AUC': baseline_auc
    },
    {
        'Library': 'sklearn',
        'Algorithm': 'Random Forest',
        'Hyperparameters': str(rf_grid.best_params_),
        'Accuracy': rf_acc,
        'AUC': rf_auc
    },
    {
        'Library': 'sklearn',
        'Algorithm': 'MLPClassifier',
        'Hyperparameters': str(mlp_grid.best_params_),
        'Accuracy': mlp_acc,
        'AUC': mlp_auc
    },
    {
        'Library': 'Keras',
        'Algorithm': 'Neural Network (2 hidden layers, dropout)',
        'Hyperparameters': "hidden_layers=(128,64), dropout=0.3, lr=0.001, class_weight=balanced",
        'Accuracy': nn_acc,
        'AUC': nn_auc
    },
    {
        'Library': 'TensorFlow (low-level)',
        'Algorithm': 'Neural Network (2 hidden layers, dropout)',
        'Hyperparameters': "hidden_layers=(128,64), dropout=0.3, lr=0.001, class_weight=balanced",
        'Accuracy': tf_acc,
        'AUC': tf_auc
    },
    {
        'Library': 'NumPy (from scratch)',
        'Algorithm': 'Neural Network (2 hidden layers, dropout, manual backprop)',
        'Hyperparameters': "hidden_layers=(128,64), dropout=0.3, lr=0.001, class_weight=balanced",
        'Accuracy': np_acc,
        'AUC': np_auc
    },
])

results = results.sort_values('AUC', ascending=False).reset_index(drop=True)
results

,Library,Algorithm,Hyperparameters,Accuracy,AUC
0,sklearn,Random Forest,"{'class_weight': 'balanced', 'max_depth': None...",0.897351,0.835626
1,NumPy (from scratch),"Neural Network (2 hidden layers, dropout, manu...","hidden_layers=(128,64), dropout=0.3, lr=0.001,...",0.677637,0.826720
2,TensorFlow (low-level),"Neural Network (2 hidden layers, dropout)","hidden_layers=(128,64), dropout=0.3, lr=0.001,...",0.646302,0.826307
3,Keras,"Neural Network (2 hidden layers, dropout)","hidden_layers=(128,64), dropout=0.3, lr=0.001,...",0.662702,0.822398
4,sklearn,MLPClassifier,"{'alpha': 0.0001, 'hidden_layer_sizes': (64,),...",0.918551,0.800384
5,sklearn,DummyClassifier (baseline),strategy='stratified',0.851446,0.500171
